In [1]:
from pathlib import Path
import json
import cv2
import base64

from ultralytics import YOLOE


# ============================================================
# НАСТРОЙКИ
# ============================================================

INPUT_DIR = Path(r"Детекция велосипедов и электровелосипедов на основе изображений")
OUTPUT_DIR = Path(r"C:\Users\Ева\Desktop\Хакатон\YoloE\Voskhod")

MODEL_PATH = r"C:\Users\Ева\Desktop\Хакатон\YoloE\yoloe-26m-seg.pt"

# ------------------------------------------------------------
# ПРОМТЫ ДЛЯ МОДЕЛИ (то, что "видит" YOLOE)
# ------------------------------------------------------------
CLASSES = [
    "a bicycle"
]

# ------------------------------------------------------------
# ИМЯ КЛАССА ДЛЯ LABELME (то, что попадёт в "label")
# ------------------------------------------------------------
# Может отличаться от промта. Если None — используется значение
# из CLASSES по индексу класса.
LABEL_NAME = "bicycle"   # <-- здесь указываем нужное имя

CONF = 0.25
IMGSZ = 640
DEVICE = 0

# Сохранять ли изображение в base64 внутри JSON (как в примере LabelMe)
INCLUDE_IMAGE_DATA = True   # True — как в примере; False — imageData: null


# ============================================================
# ПАПКИ РЕЗУЛЬТАТОВ
# ============================================================

OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_JSON = OUTPUT_DIR / "json"

OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.mkdir(parents=True, exist_ok=True)


# ============================================================
# ЗАГРУЗКА МОДЕЛИ
# ============================================================

print("Loading model...")

model = YOLOE(MODEL_PATH)
model.set_classes(CLASSES)

print("Model loaded:", MODEL_PATH)
print("\nClasses (prompts):")
for i, name in enumerate(CLASSES):
    print(f"  {i}: {name}")

print(f"\nLabelMe label for boxes: {LABEL_NAME!r}")


# ============================================================
# ПОИСК ИЗОБРАЖЕНИЙ
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

image_files = sorted(
    p for p in INPUT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"\nFound {len(image_files)} images")


# ============================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def encode_image_base64(image_path: Path) -> str:
    """Кодирует файл изображения в base64-строку для поля imageData."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_label_for_class(class_id: int) -> str:
    """
    Возвращает имя класса для LabelMe.
    Если LABEL_NAME задан — используем его для всех боксов.
    Иначе — берём из CLASSES по индексу.
    """
    if LABEL_NAME is not None:
        return LABEL_NAME
    if 0 <= class_id < len(CLASSES):
        return CLASSES[class_id]
    return str(class_id)


# ============================================================
# ОБРАБОТКА ИЗОБРАЖЕНИЯ
# ============================================================

def process_image(image_path: Path):

    relative_path = image_path.relative_to(INPUT_DIR)
    relative_parent = relative_path.parent

    image_output_dir = OUTPUT_IMAGES / relative_parent
    json_output_dir = OUTPUT_JSON / relative_parent

    image_output_dir.mkdir(parents=True, exist_ok=True)
    json_output_dir.mkdir(parents=True, exist_ok=True)

    output_image_path = image_output_dir / image_path.name
    output_json_path = json_output_dir / f"{image_path.stem}.json"

    # --------------------------------------------------------
    # Читаем изображение
    # --------------------------------------------------------

    image = cv2.imread(str(image_path))

    if image is None:
        print(f"[ERROR] Cannot read: {image_path}")
        return

    height, width = image.shape[:2]

    # --------------------------------------------------------
    # YOLOE inference
    # --------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        conf=CONF,
        imgsz=IMGSZ,
        device=DEVICE,
        verbose=False
    )

    result = results[0]

    # --------------------------------------------------------
    # LabelMe JSON — каркас
    # --------------------------------------------------------

    image_path_for_json = f"..\\\\images\\\\{image_path.name}"

    image_data_value = None
    if INCLUDE_IMAGE_DATA:
        try:
            image_data_value = encode_image_base64(image_path)
        except Exception as e:
            print(f"[WARN] base64 encode failed for {image_path}: {e}")
            image_data_value = None

    labelme_data = {
        "version": "5.2.1",
        "flags": {},
        "shapes": [],
        "imagePath": image_path_for_json,
        "imageData": image_data_value,
        "imageHeight": int(height),
        "imageWidth": int(width),
    }

    annotated = image.copy()

    # --------------------------------------------------------
    # Нет детекций
    # --------------------------------------------------------

    if result.boxes is None or len(result.boxes) == 0:
        cv2.imwrite(str(output_image_path), annotated)

        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(labelme_data, f, ensure_ascii=False, indent=2)

        print(f"[NO DETECTIONS] {relative_path}")
        return

    # --------------------------------------------------------
    # Получаем bbox
    # --------------------------------------------------------

    boxes = result.boxes.xyxy.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)
    confidences = result.boxes.conf.cpu().numpy()

    # --------------------------------------------------------
    # Каждый объект
    # --------------------------------------------------------

    for box, class_id, confidence in zip(boxes, class_ids, confidences):

        x1, y1, x2, y2 = box

        class_id = int(class_id)
        confidence = float(confidence)

        # Ограничиваем координаты
        x1 = max(0.0, min(width - 1, float(x1)))
        y1 = max(0.0, min(height - 1, float(y1)))
        x2 = max(0.0, min(width - 1, float(x2)))
        y2 = max(0.0, min(height - 1, float(y2)))

        # ----------------------------------------------------
        # Имя класса для label в LabelMe
        # (может отличаться от промта YOLOE)
        # ----------------------------------------------------
        class_name = get_label_for_class(class_id)

        # ====================================================
        # LABELME SHAPE
        # ====================================================

        shape = {
            "label": class_name,
            "points": [
                [float(x1), float(y1)],
                [float(x2), float(y2)],
            ],
            "group_id": None,
            "description": "",
            "shape_type": "rectangle",
            "flags": {},
        }

        labelme_data["shapes"].append(shape)

        # ====================================================
        # ВИЗУАЛИЗАЦИЯ
        # ====================================================

        cv2.rectangle(
            annotated,
            (int(x1), int(y1)),
            (int(x2), int(y2)),
            (0, 255, 0),
            2
        )

        text = f"{class_name} {confidence:.2f}"

        (text_width, text_height), baseline = cv2.getTextSize(
            text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
        )

        text_x = int(x1)
        text_y = max(int(y1), text_height + baseline)

        cv2.rectangle(
            annotated,
            (text_x, text_y - text_height - baseline),
            (text_x + text_width, text_y),
            (0, 255, 0),
            -1
        )

        cv2.putText(
            annotated,
            text,
            (text_x, text_y - baseline),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 0),
            2,
            cv2.LINE_AA
        )

    # ========================================================
    # СОХРАНЕНИЕ
    # ========================================================

    cv2.imwrite(str(output_image_path), annotated)

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(labelme_data, f, ensure_ascii=False, indent=2)

    print(f"[OK] {relative_path} | detections: {len(boxes)}")


# ============================================================
# ЗАПУСК
# ============================================================

print("\nStarting inference...\n")

for i, image_path in enumerate(image_files, 1):
    print(f"[{i}/{len(image_files)}] {image_path.name}")
    try:
        process_image(image_path)
    except Exception as e:
        print(f"[ERROR] {image_path}")
        print(f"       {type(e).__name__}: {e}")


# ============================================================
# ГОТОВО
# ============================================================

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print(f"Visualized images: {OUTPUT_IMAGES}")
print(f"LabelMe JSON:       {OUTPUT_JSON}")

Loading model...


c:\Users\Ева\Desktop\Хакатон\venv\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Model loaded: C:\Users\Ева\Desktop\Хакатон\YoloE\yoloe-26m-seg.pt

Classes (prompts):
  0: a bicycle

LabelMe label for boxes: 'bicycle'

Found 553 images

Starting inference...

[1/553] 000001.jpg
[NO DETECTIONS] images\000001.jpg
[2/553] 000002.png
[OK] images\000002.png | detections: 1
[3/553] 000003.jpg
[NO DETECTIONS] images\000003.jpg
[4/553] 000004.png
[NO DETECTIONS] images\000004.png
[5/553] 000005.jpg
[NO DETECTIONS] images\000005.jpg
[6/553] 000006.png
[OK] images\000006.png | detections: 1
[7/553] 000007.jpg
[NO DETECTIONS] images\000007.jpg
[8/553] 000008.png
[NO DETECTIONS] images\000008.png
[9/553] 000009.jpg
[NO DETECTIONS] images\000009.jpg
[10/553] 000010.png
[OK] images\000010.png | detections: 1
[11/553] 000011.jpg
[NO DETECTIONS] images\000011.jpg
[12/553] 000012.png
[NO DETECTIONS] images\000012.png
[13/553] 000013.jpg
[NO DETECTIONS] images\000013.jpg
[14/553] 000014.png
[NO DETECTIONS] images\000014.png
[15/553] 000015.jpg
[NO DETECTIONS] images\000015.jpg
[16/5